In [3]:
# 1) Imports
import pandas as pd
import plotly.graph_objects as go
import unicodedata

In [4]:
# 2) Definição de variáveis / parâmetros
CSV_MORTALIDADE = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"

# Filtros (cada linha do CSV = um município)
MUNICIPIO        = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE   = None

# Intervalo de anos
ANOS_RANGE    = (1979, 2024)
ANOS_PASSO    = 1

# Faixas etárias (barras)
FAIXAS_ETARIAS = [
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70 ou mais",
]

# Cores e título
COR_BARRAS = "#2D9CDB"
COR_PONTOS = "#D7263D"
TITULO     = "Mortalidade por faixa etária (barras) + total por ano (pontos)"

In [5]:
# 3) Ajustes / tratamento dos dados (leitura, filtro, somatórios)
def _norm_txt(x):
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    return "".join(ch for ch in x if not unicodedata.combining(ch))


def _norm_series(s):
    return s.astype(str).map(_norm_txt)


def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)


def _col_ano_faixa(ano, faixa):
    return f"{int(ano)}_{faixa}"


def _col_total_ano(ano):
    return f"Total_{int(ano)}"


def _sum_row_wise(frame, cols):
    """Replica df[col1] + df[col2] + ... (por linha)."""
    if not cols:
        return pd.Series([0] * len(frame), index=frame.index, dtype="int64")
    return (
        frame[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
        .round()
        .astype(int)
    )


df = df_mortalidade.copy() if "df_mortalidade" in globals() else pd.read_csv(CSV_MORTALIDADE)
f = df.copy()

if MUNICIPIO is not None:
    f = f[_norm_series(f["Municípios"]) == _norm_txt(MUNICIPIO)]
if CODIGO_MUNICIPIO is not None:
    f = f[pd.to_numeric(f["Código"], errors="coerce").astype("Int64") == int(CODIGO_MUNICIPIO)]
if REGIONAL_SAUDE is not None:
    f = f[_norm_series(f["Regional de Saúde"]) == _norm_txt(REGIONAL_SAUDE)]

if f.empty:
    raise ValueError("Filtro não encontrou nenhuma linha.")

print(f"Linhas selecionadas: {len(f)}")

ano_ini, ano_fim = map(int, ANOS_RANGE)
anos = list(range(ano_ini, ano_fim + 1))

# Barras: soma por faixa etária no intervalo
por_faixa = {}
for fx in FAIXAS_ETARIAS:
    cols = [_col_ano_faixa(a, fx) for a in anos if _col_ano_faixa(a, fx) in f.columns]
    por_faixa[fx] = int(_sum_row_wise(f, cols).sum())

# Pontos: total por ano
por_ano = {}
for ano in anos:
    col_total = _col_total_ano(ano)
    if col_total in f.columns:
        por_ano[ano] = int(pd.to_numeric(f[col_total], errors="coerce").fillna(0).sum())
    else:
        cols = [_col_ano_faixa(ano, fx) for fx in FAIXAS_ETARIAS if _col_ano_faixa(ano, fx) in f.columns]
        por_ano[ano] = int(_sum_row_wise(f, cols).sum())

por_ano_s = pd.Series(por_ano).sort_index()

print("Somatório por faixa etária:")
for fx, v in por_faixa.items():
    print(f"  {fx}: {_fmt_int(v)}")
print(f"\nTotal geral (faixas): {_fmt_int(sum(por_faixa.values()))}")

Linhas selecionadas: 841
Somatório por faixa etária:
  0-9: 1
  10-19: 11
  20-29: 325
  30-39: 2.977
  40-49: 7.385
  50-59: 10.214
  60-69: 9.345
  70 ou mais: 12.164

Total geral (faixas): 42.422


In [6]:
# 4) Design e layout do gráfico (Plotly)
fig = go.Figure()

# ── Barras horizontais ──────────────────────────────────────────
valores_barras = [por_faixa[fx] for fx in FAIXAS_ETARIAS]
tooltips_barras = [
    f"<b>Faixa etária:</b> {fx}<br><b>Total de mortes:</b> {_fmt_int(por_faixa[fx])}<br><i>Soma {ano_ini}–{ano_fim}</i>"
    for fx in FAIXAS_ETARIAS
]

fig.add_trace(go.Bar(
    name=f"Total por faixa etária ({ano_ini}–{ano_fim})",
    x=valores_barras,
    y=FAIXAS_ETARIAS,
    orientation="h",
    marker=dict(
        color=COR_BARRAS,
        opacity=0.85,
        line=dict(color="white", width=0.8),
    ),
    text=[_fmt_int(v) for v in valores_barras],
    textposition="outside",
    textfont=dict(size=11, color="#222"),
    hovertext=tooltips_barras,
    hoverinfo="text",
    hovertemplate="%{hovertext}<extra></extra>",
    yaxis="y1",
    xaxis="x",
))

# ── Pontos por ano ───────────────────────────────────────────────
anos_lista   = por_ano_s.index.tolist()
mortes_lista = por_ano_s.values.tolist()
tooltips_pontos = [
    f"<b>Ano:</b> {ano}<br><b>Total de mortes:</b> {_fmt_int(val)}"
    for ano, val in zip(anos_lista, mortes_lista)
]

fig.add_trace(go.Scatter(
    name=f"Total por ano ({ano_ini}–{ano_fim})",
    x=mortes_lista,
    y=anos_lista,
    mode="markers+text",
    marker=dict(
        color=COR_PONTOS,
        size=9,
        opacity=0.92,
        line=dict(color="white", width=1.2),
        symbol="circle",
    ),
    text=[_fmt_int(v) for v in mortes_lista],
    textposition="middle right",
    textfont=dict(size=9, color=COR_PONTOS),
    hovertext=tooltips_pontos,
    hoverinfo="text",
    hovertemplate="%{hovertext}<extra></extra>",
    yaxis="y2",
    xaxis="x",
))

# ── Layout ───────────────────────────────────────────────────────
x_max_data = max(max(valores_barras), max(mortes_lista))
step = 500 if x_max_data > 5000 else (200 if x_max_data > 2000 else 100)
xticks_vals = list(range(1, 11)) + list(range(step, int(x_max_data * 1.15) + step, step))
xticks_vals = sorted(set(v for v in xticks_vals if v <= int(x_max_data * 1.15)))

fig.update_layout(
    title=dict(
        text=TITULO,
        font=dict(size=16, family="Arial", color="#1a1a2e"),
        x=0.5, xanchor="center",
        pad=dict(b=20),
    ),
    height=620,
    width=1400,
    plot_bgcolor="#f8f9fb",
    paper_bgcolor="white",
    margin=dict(l=80, r=180, t=80, b=80),
    hovermode="closest",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.03,
        xanchor="center", x=0.5,
        font=dict(size=11),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#ddd", borderwidth=1,
    ),
    # Eixo X compartilhado
    xaxis=dict(
        title="Número de mortes",
        tickvals=xticks_vals,
        ticktext=[_fmt_int(v) for v in xticks_vals],
        tickfont=dict(size=10),
        title_font=dict(size=12),
        gridcolor="#dde3ed",
        gridwidth=1,
        showline=True,
        linecolor="#ccc",
        zeroline=True,
        zerolinecolor="#bbb",
        range=[0, x_max_data * 1.25],
    ),
    # Eixo Y esquerdo — faixas etárias
    yaxis=dict(
        title="Faixa etária",
        title_font=dict(size=12),
        tickfont=dict(size=11),
        showgrid=False,
        showline=True,
        linecolor="#ccc",
        domain=[0, 1],
    ),
    # Eixo Y direito — anos
    yaxis2=dict(
        title="Ano",
        title_font=dict(size=12),
        tickfont=dict(size=10),
        overlaying="y",
        side="right",
        tickvals=list(range(ano_ini, ano_fim + 1, ANOS_PASSO)),
        showgrid=False,
        showline=True,
        linecolor="#ccc",
        range=[ano_ini - 0.5, ano_fim + 0.5],
    ),
    # Hoverlabel visual
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#aaa",
        font=dict(size=12, family="Arial"),
    ),
)

# Grade horizontal suave só nas barras
fig.update_layout(
    shapes=[
        dict(
            type="line",
            x0=0, x1=1, xref="paper",
            y0=i, y1=i, yref="y",
            line=dict(color="#dde3ed", width=1, dash="dot"),
        )
        for i, _ in enumerate(FAIXAS_ETARIAS)
    ]
)

fig.show()
print("✅ Gráfico Plotly exibido — passe o mouse nas barras e bolinhas para ver o tooltip.")

✅ Gráfico Plotly exibido — passe o mouse nas barras e bolinhas para ver o tooltip.
